# Dataset Inspection

This notebook performs initial exploratory analysis of the Customer Support on Twitter dataset.

**Note:** Run `python scripts/download_dataset.py` first to acquire the dataset.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'

## 1. Load Dataset

In [ ]:
# Find the main CSV file
csv_files = list(RAW_DIR.glob('*.csv'))
if not csv_files:
    print('No CSV files found. Run: python scripts/download_dataset.py')
else:
    print(f'Found CSV files: {[f.name for f in csv_files]}')
    main_csv = csv_files[0]
    print(f'Loading: {main_csv.name}')
    
    # Load in chunks for large files
    chunks = []
    for chunk in pd.read_csv(main_csv, chunksize=50000, low_memory=False):
        chunks.append(chunk)
    df = pd.concat(chunks, ignore_index=True)
    print(f'Loaded {len(df):,} rows and {len(df.columns)} columns.')

## 2. Basic Overview

In [ ]:
if 'df' in dir():
    print(f'Shape: {df.shape}')
    print(f'\nColumns: {list(df.columns)}')
    print(f'\nData types:\n{df.dtypes}')
    print(f'\nFirst 3 rows:')
    df.head(3)

## 3. Missingness Analysis

In [ ]:
if 'df' in dir():
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        'missing_count': missing,
        'missing_pct': missing_pct
    }).sort_values('missing_pct', ascending=False)
    
    # Only show columns with missing values
    missing_df = missing_df[missing_df['missing_count'] > 0]
    
    if len(missing_df) > 0:
        print('Columns with missing values:')
        display(missing_df)
        
        # Plot
        fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df) * 0.5)))
        missing_df['missing_pct'].plot(kind='barh', ax=ax)
        ax.set_xlabel('Missing %')
        ax.set_title('Missing Values by Column')
        plt.tight_layout()
        plt.show()
    else:
        print('No missing values found.')

## 4. Duplicate Analysis

In [ ]:
if 'df' in dir():
    exact_dups = df.duplicated().sum()
    print(f'Exact duplicate rows: {exact_dups:,} ({exact_dups/len(df)*100:.2f}%)')
    
    # Check for ID-like columns
    id_cols = [col for col in df.columns if 'id' in col.lower() or 'tweet' in col.lower()]
    if id_cols:
        print(f'\nID-like columns: {id_cols}')
        for col in id_cols:
            if col in df.columns:
                n_unique = df[col].nunique()
                n_dups = len(df) - n_unique
                print(f'  {col}: {n_unique:,} unique, {n_dups:,} duplicates')

## 5. Text Quality Analysis

In [ ]:
if 'df' in dir():
    # Find text column
    text_col = None
    for col in ['text', 'message', 'content', 'tweet_text']:
        if col in df.columns:
            text_col = col
            break
    
    if text_col:
        print(f'Text column: {text_col}')
        
        # Compute text lengths
        df['text_length'] = df[text_col].astype(str).str.len()
        
        # Statistics
        print(f'\nText length statistics:')
        print(df['text_length'].describe())
        
        # Empty messages
        empty_count = (df[text_col].isna() | (df[text_col].astype(str).str.strip() == '')).sum()
        print(f'\nEmpty messages: {empty_count:,} ({empty_count/len(df)*100:.2f}%)')
        
        # Very short messages (< 10 chars)
        short_count = (df['text_length'] < 10).sum()
        print(f'Very short (< 10 chars): {short_count:,} ({short_count/len(df)*100:.2f}%)')
        
        # Very long messages (> 280 chars)
        long_count = (df['text_length'] > 280).sum()
        print(f'Very long (> 280 chars): {long_count:,} ({long_count/len(df)*100:.2f}%)')
        
        # Plot distribution
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Histogram
        df['text_length'].clip(upper=500).hist(bins=50, ax=axes[0], edgecolor='black')
        axes[0].set_xlabel('Text Length (chars)')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Message Length Distribution')
        axes[0].axvline(df['text_length'].median(), color='red', linestyle='--', label=f'Median: {df["text_length"].median():.0f}')
        axes[0].legend()
        
        # Box plot
        df['text_length'].clip(upper=500).plot(kind='box', ax=axes[1])
        axes[1].set_ylabel('Text Length (chars)')
        axes[1].set_title('Message Length Box Plot')
        
        plt.tight_layout()
        plt.show()
    else:
        print('No text column identified.')
        print(f'Available columns: {list(df.columns)}')

## 6. Temporal Analysis

In [ ]:
if 'df' in dir():
    # Find timestamp column
    time_col = None
    for col in ['created_at', 'timestamp', 'date', 'time', 'datetime']:
        if col in df.columns:
            time_col = col
            break
    
    if time_col:
        print(f'Timestamp column: {time_col}')
        
        # Parse timestamps
        df['timestamp'] = pd.to_datetime(df[time_col], errors='coerce')
        valid_ts = df['timestamp'].dropna()
        
        if len(valid_ts) > 0:
            print(f'Earliest: {valid_ts.min()}')
            print(f'Latest: {valid_ts.max()}')
            print(f'Date range: {(valid_ts.max() - valid_ts.min()).days} days')
            
            # Messages per month
            df['year_month'] = df['timestamp'].dt.to_period('M')
            monthly = df.groupby('year_month').size()
            
            fig, ax = plt.subplots(figsize=(14, 5))
            monthly.plot(ax=ax)
            ax.set_xlabel('Month')
            ax.set_ylabel('Number of Messages')
            ax.set_title('Messages Over Time')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        else:
            print('No valid timestamps found.')
    else:
        print('No timestamp column identified.')

## 7. Conversation Analysis

In [ ]:
if 'df' in dir():
    # Find conversation column
    conv_col = None
    for col in ['conversation_id', 'thread_id', 'in_reply_to_tweet_id']:
        if col in df.columns:
            conv_col = col
            break
    
    if conv_col:
        print(f'Conversation column: {conv_col}')
        
        n_conversations = df[conv_col].nunique()
        print(f'Unique conversations: {n_conversations:,}')
        
        # Conversation length distribution
        conv_lengths = df.groupby(conv_col).size()
        
        print(f'\nConversation length statistics:')
        print(conv_lengths.describe())
        
        # Single vs multi-message conversations
        single_msg = (conv_lengths == 1).sum()
        multi_msg = (conv_lengths > 1).sum()
        print(f'\nSingle-message conversations: {single_msg:,} ({single_msg/n_conversations*100:.1f}%)')
        print(f'Multi-message conversations: {multi_msg:,} ({multi_msg/n_conversations*100:.1f}%)')
        
        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Length distribution
        conv_lengths.clip(upper=50).hist(bins=50, ax=axes[0], edgecolor='black')
        axes[0].set_xlabel('Messages per Conversation')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Conversation Length Distribution')
        axes[0].axvline(conv_lengths.median(), color='red', linestyle='--', label=f'Median: {conv_lengths.median():.0f}')
        axes[0].legend()
        
        # Single vs multi
        labels = ['Single', 'Multi']
        sizes = [single_msg, multi_msg]
        axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
        axes[1].set_title('Conversation Type')
        
        plt.tight_layout()
        plt.show()
    else:
        print('No conversation column identified.')

## 8. Brand Analysis

In [ ]:
if 'df' in dir():
    # Find brand column
    brand_col = None
    for col in ['brand', 'company', 'author_name', 'inbound_author']:
        if col in df.columns:
            brand_col = col
            break
    
    if brand_col:
        print(f'Brand column: {brand_col}')
        
        n_brands = df[brand_col].nunique()
        print(f'Unique brands: {n_brands}')
        
        # Top brands by message count
        brand_counts = df[brand_col].value_counts().head(20)
        
        print(f'\nTop 20 brands by message count:')
        display(brand_counts)
        
        # Plot
        fig, ax = plt.subplots(figsize=(12, 8))
        brand_counts.plot(kind='barh', ax=ax)
        ax.set_xlabel('Number of Messages')
        ax.set_ylabel('Brand')
        ax.set_title('Top 20 Brands by Message Count')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
    else:
        print('No brand column identified.')
        print(f'Available columns: {list(df.columns)}')

## 9. Save Statistics

In [ ]:
if 'df' in dir():
    stats = {
        'rows': len(df),
        'columns': len(df.columns),
        'column_names': list(df.columns),
        'missingness': {col: int(df[col].isnull().sum()) for col in df.columns},
    }
    
    # Add text statistics if available
    if 'text_col' in dir() and text_col:
        stats['text_statistics'] = {
            'mean_length': float(df['text_length'].mean()),
            'median_length': float(df['text_length'].median()),
            'min_length': int(df['text_length'].min()),
            'max_length': int(df['text_length'].max()),
        }
    
    # Add conversation statistics if available
    if 'conv_col' in dir() and conv_col:
        conv_lengths = df.groupby(conv_col).size()
        stats['conversation_statistics'] = {
            'unique_conversations': int(df[conv_col].nunique()),
            'mean_conversation_length': float(conv_lengths.mean()),
            'median_conversation_length': float(conv_lengths.median()),
        }
    
    # Add brand statistics if available
    if 'brand_col' in dir() and brand_col:
        stats['brand_statistics'] = {
            'unique_brands': int(df[brand_col].nunique()),
        }
    
    # Save
    output_path = INTERIM_DIR / 'dataset_statistics.json'
    with open(output_path, 'w') as f:
        json.dump(stats, f, indent=2, default=str)
    print(f'Statistics saved to: {output_path}')
    print(json.dumps(stats, indent=2, default=str))